In [0]:
CATALOG_NAME = "subscription_pipeline"
BRONZE_SCHEMA = "bronze"

# Create catalog & schema
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{BRONZE_SCHEMA}")

# Source schema (Fivetran)
fivetran_source_schema = "analytics.azure_blob_storage"

tables_to_process = [
    "country_master",
    "customer",
    "employee",
    "fx_rate",
    "opportunity",
    "product"
]

from pyspark.sql.functions import current_timestamp

for table in tables_to_process:
    print(f"Ingesting {table} into Bronze layer...")

    # Read raw data from Fivetran
    df_raw = spark.read.table(f"{fivetran_source_schema}.{table}")

    # Add only ingestion timestamp (allowed in Bronze)
    df_bronze = df_raw.withColumn("ingestion_timestamp", current_timestamp())

    # Target Bronze table
    target_table = f"{CATALOG_NAME}.{BRONZE_SCHEMA}.{table}"

    # Append data (DO NOT overwrite)
    df_bronze.write.format("delta")\
        .mode("overwrite") \
             .option("overwriteSchema", "true") \
                 .saveAsTable(target_table)   

print("Bronze ingestion completed successfully!")